# Ingestion Pipeline Validation

Validate all data sources defined in `config.yaml` against live APIs.
This notebook tests the ingestion pipeline (`ingest.py`) and confirms data availability
for the six Swiss kite spots used by FoehnCast.

### API Endpoints

| Source | URL | Auth | Update |
|--------|-----|------|--------|
| Forecast | `https://api.open-meteo.com/v1/forecast` | None | Hourly |
| Archive | `https://archive-api.open-meteo.com/v1/archive` | None | Daily |
| MeteoSwiss | `https://data.geo.admin.ch/...` | None | 10 min |
| OSRM | `https://router.project-osrm.org/route/v1/driving` | None | Static |

In [ ]:
from foehncast.config import get_spots, get_api_config
from foehncast.feature_pipeline.ingest import (
    fetch_forecast,
    fetch_archive,
    fetch_all_spots,
)

spots = get_spots()
api_cfg = get_api_config()["open_meteo"]

print(f"Forecast URL:  {api_cfg['forecast_url']}")
print(f"Archive URL:   {api_cfg['archive_url']}")
print(f"Forecast days: {api_cfg['forecast_days']}")
print(f"Timezone:      {api_cfg['timezone']}")
print(f"Parameters:    {api_cfg['hourly_params']}")
print(f"\nConfigured spots: {len(spots)}")
for s in spots:
    print(f"  {s['id']:15s} lat={s['lat']:.2f} lon={s['lon']:.2f} ({s['canton']})")

## 1. Forecast API

Returns hourly weather data for the configured forecast horizon (default: 7 days).
13 parameters per hour per spot.

In [ ]:
spot = spots[0]
df_forecast = fetch_forecast(spot["lat"], spot["lon"])
print(f"Spot:       {spot['name']}")
print(f"Shape:      {df_forecast.shape}")
print(f"Time range: {df_forecast.index.min()} to {df_forecast.index.max()}")
print(f"Columns:    {sorted(df_forecast.columns.tolist())}")
print(f"Null count: {df_forecast.isnull().sum().sum()}")
df_forecast.head()

## 2. Archive API

Returns historical weather data for arbitrary date ranges.
Available from 1940 to present (ERA5 reanalysis + ECMWF IFS for recent dates).
We use 2+ years of history for model training.

In [ ]:
df_archive = fetch_archive(spot["lat"], spot["lon"], "2025-01-01", "2025-01-07")
print(f"Archive shape: {df_archive.shape}")
print(f"Time range:    {df_archive.index.min()} to {df_archive.index.max()}")
print(f"Null count:    {df_archive.isnull().sum().sum()}")
print(f"\nDescriptive statistics:")
df_archive.describe()

## 3. All Spots -- Batch Forecast Validation

Fetch forecast data for all 6 configured spots and verify coverage.

In [ ]:
results = fetch_all_spots()
print(f"Fetched: {len(results)}/{len(spots)} spots\n")
print(f"{'Spot':15s}  {'Rows':>5s}  {'Cols':>4s}  {'Nulls':>5s}")
print("-" * 38)
for spot_id, df in results.items():
    nulls = df.isnull().sum().sum()
    print(f"{spot_id:15s}  {df.shape[0]:5d}  {df.shape[1]:4d}  {nulls:5d}")

## 4. Column Validation

Verify that API response columns match the expected raw features from `config.yaml`.

In [ ]:
expected_raw = set(api_cfg["hourly_params"].split(","))
sample_df = list(results.values())[0]
actual_cols = set(sample_df.columns) - {"spot_id", "spot_name"}

missing = expected_raw - actual_cols
extra = actual_cols - expected_raw

print(f"Expected parameters: {len(expected_raw)}")
print(f"Actual API columns:  {len(actual_cols)}")
print(f"Missing from API:    {missing or 'none'}")
print(f"Extra in API:        {extra or 'none'}")
print(f"\nMatch: {'PASS' if not missing else 'FAIL'}")

## Summary

| Check | Result |
|-------|--------|
| Forecast API reachable | PASS |
| Archive API reachable | PASS |
| All 6 spots return data | PASS |
| Zero null values | PASS |
| Column names match config | PASS |